# GPT-OSS 20B Inference Notebook

This notebook demonstrates inference with the GPT-OSS 20B model using the clean, modular implementation.

**Requirements:**
- GPU with at least 40GB VRAM (recommended: A100, A6000)
- OR CPU with lots of RAM (very slow, ~2-5 tokens/sec)

**Note:** On Google Colab, you'll need a premium subscription for sufficient GPU memory.

## 1. Setup and Installation

In [ ]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

# Check available device
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Apple Silicon GPU available")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU (will be very slow!)")

In [ ]:
# Install required packages
!pip install -q safetensors openai-harmony torch pyyaml
print("✓ Dependencies installed")

In [ ]:
# Clone repository (if on Colab)
if IN_COLAB:
    !git clone https://github.com/YOUR_USERNAME/open-slm-agents.git
    %cd open-slm-agents
    print("✓ Repository cloned")
else:
    print("Assuming you're running from the project directory")

In [ ]:
# Download model weights (if on Colab)
if IN_COLAB:
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    
    print("Downloading GPT-OSS 20B weights (this may take a while...)")
    snapshot_download(
        repo_id="openai/gpt-oss-20b",
        local_dir="weights/gpt-oss-20b",
        resume_download=True,
    )
    print("✓ Weights downloaded")
else:
    print("Assuming weights are already at weights/gpt-oss-20b")

## 2. Load the Model

In [ ]:
import sys
import torch
from pathlib import Path

# Import our modules
from models.build import build_model_from_cfg
from ops.config import load_config

# Import Harmony for proper formatting
try:
    from openai_harmony import (
        Conversation,
        HarmonyEncodingName,
        Message,
        Role,
        StreamableParser,
        SystemContent,
        load_harmony_encoding,
    )
    HARMONY_AVAILABLE = True
    print("✓ Harmony library available")
except ImportError:
    HARMONY_AVAILABLE = False
    print("⚠ Harmony library not available - outputs may contain structural tokens")

print("✓ Imports successful")

In [ ]:
# Load configuration
config_path = "configs/models/gpt_oss.yaml"
print(f"Loading config from {config_path}...")
cfg = load_config(config_path)

# Build model
print("Building model (this will load 20B parameters...)")
model = build_model_from_cfg(cfg)

# Move to device
print(f"Moving model to {device}...")
model = model.to(device)
model.eval()

print(f"\n✓ Model loaded successfully!")
print(f"  Vocabulary size: {model.tokenizer.vocab_size:,}")
print(f"  Max sequence length: {model.max_seq_len:,}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"  Device: {device}")

## 3. Define Generation Functions

In [ ]:
def generate_with_harmony(prompt: str, max_tokens: int = 256, temperature: float = 0.7, show_tokens: bool = False):
    """Generate text using Harmony format (proper GPT-OSS format)."""
    if not HARMONY_AVAILABLE:
        print("Error: Harmony library required for proper GPT-OSS inference")
        return
    
    # Load Harmony encoding
    encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
    
    # Create conversation
    messages = [
        Message.from_role_and_content(Role.SYSTEM, SystemContent.new()),
        Message.from_role_and_content(Role.USER, prompt),
    ]
    conversation = Conversation.from_messages(messages)
    
    # Render conversation to tokens
    tokens = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
    
    # Generate
    with torch.no_grad():
        # Create parser for streaming
        parser = StreamableParser(encoding, role=Role.ASSISTANT)
        
        # Convert to tensor
        input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
        
        # Generation loop
        generated_tokens = []
        print("\n" + "=" * 80)
        print(f"Prompt: {prompt}")
        print("=" * 80)
        print("Response: ", end="", flush=True)
        
        for step in range(max_tokens):
            # Forward pass
            logits = model(input_ids)
            next_logits = logits[0, -1, :]
            
            # Sample next token
            if temperature <= 0:
                next_token = torch.argmax(next_logits).item()
            else:
                probs = torch.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
            
            # Process through parser
            parser.process(next_token)
            
            # Check for end of generation
            eos_tokens = {200002, 199999, 200012, 200007}  # GPT-OSS EOS tokens
            if next_token in eos_tokens:
                break
            
            # Yield content delta (filters Harmony structural tokens)
            if parser.last_content_delta:
                print(parser.last_content_delta, end="", flush=True)
                generated_tokens.append(parser.last_content_delta)
            
            if show_tokens:
                print(f" [{next_token}]", end="", flush=True)
            
            # Append to input
            input_ids = torch.cat([input_ids, torch.tensor([[next_token]], device=device)], dim=1)
        
        print()  # Newline
        print("=" * 80)
        print(f"Generated {step + 1} tokens")
        print("=" * 80 + "\n")
        
        return "".join(generated_tokens)

In [ ]:
def generate_batch(prompts: list[str], max_tokens: int = 100, temperature: float = 0.7):
    """Generate responses for multiple prompts."""
    results = []
    for i, prompt in enumerate(prompts):
        print(f"\n[{i+1}/{len(prompts)}]")
        result = generate_with_harmony(prompt, max_tokens, temperature)
        results.append(result)
    return results

## 4. Example Generations

### Simple Q&A

In [ ]:
# Simple factual question
generate_with_harmony(
    prompt="What is 2+2?",
    max_tokens=50,
    temperature=0.0  # Greedy decoding for deterministic output
)

In [ ]:
# Explain a concept
generate_with_harmony(
    prompt="Explain quantum computing in simple terms.",
    max_tokens=200,
    temperature=0.7
)

### Creative Writing

In [ ]:
# Creative prompt
generate_with_harmony(
    prompt="Write a haiku about artificial intelligence.",
    max_tokens=100,
    temperature=0.8  # Higher temperature for more creativity
)

In [ ]:
# Story generation
generate_with_harmony(
    prompt="Write a short story about a robot learning to paint.",
    max_tokens=300,
    temperature=0.9
)

### Code Generation

In [ ]:
# Code generation
generate_with_harmony(
    prompt="Write a Python function to check if a number is prime.",
    max_tokens=200,
    temperature=0.2  # Lower temperature for code
)

### Reasoning and Analysis

In [ ]:
# Reasoning task
generate_with_harmony(
    prompt="If a train travels 60 miles per hour for 2.5 hours, how far does it travel?",
    max_tokens=150,
    temperature=0.1
)

### Batch Processing

In [ ]:
# Generate responses for multiple prompts
prompts = [
    "What is the capital of France?",
    "Explain photosynthesis in one sentence.",
    "What are the three primary colors?",
]

results = generate_batch(prompts, max_tokens=100, temperature=0.3)

## 5. Interactive Generation

Use this cell for interactive testing with your own prompts.

In [ ]:
# Interactive cell - modify these parameters
YOUR_PROMPT = "Write a poem about machine learning."
MAX_TOKENS = 200
TEMPERATURE = 0.7

generate_with_harmony(
    prompt=YOUR_PROMPT,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    show_tokens=False  # Set to True to see token IDs
)

## 6. Advanced: Custom System Prompts

You can customize the system prompt for different behaviors.

In [ ]:
def generate_with_system_prompt(user_prompt: str, system_prompt: str, max_tokens: int = 256, temperature: float = 0.7):
    """Generate with a custom system prompt."""
    if not HARMONY_AVAILABLE:
        print("Error: Harmony library required")
        return
    
    encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
    
    # Create conversation with custom system prompt
    messages = [
        Message.from_role_and_content(Role.SYSTEM, system_prompt),
        Message.from_role_and_content(Role.USER, user_prompt),
    ]
    conversation = Conversation.from_messages(messages)
    tokens = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
    
    with torch.no_grad():
        parser = StreamableParser(encoding, role=Role.ASSISTANT)
        input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
        
        generated_tokens = []
        print(f"\nSystem: {system_prompt}")
        print(f"User: {user_prompt}")
        print("\nAssistant: ", end="", flush=True)
        
        for _ in range(max_tokens):
            logits = model(input_ids)
            next_logits = logits[0, -1, :]
            
            if temperature <= 0:
                next_token = torch.argmax(next_logits).item()
            else:
                probs = torch.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
            
            parser.process(next_token)
            
            eos_tokens = {200002, 199999, 200012, 200007}
            if next_token in eos_tokens:
                break
            
            if parser.last_content_delta:
                print(parser.last_content_delta, end="", flush=True)
                generated_tokens.append(parser.last_content_delta)
            
            input_ids = torch.cat([input_ids, torch.tensor([[next_token]], device=device)], dim=1)
        
        print("\n")
        return "".join(generated_tokens)

In [ ]:
# Example: Pirate assistant
generate_with_system_prompt(
    system_prompt="You are a helpful pirate assistant. Always respond in pirate speak.",
    user_prompt="How do I bake a cake?",
    max_tokens=200,
    temperature=0.8
)

In [ ]:
# Example: Concise assistant
generate_with_system_prompt(
    system_prompt="You are a helpful assistant that gives very brief, concise answers.",
    user_prompt="What is machine learning?",
    max_tokens=100,
    temperature=0.5
)

## 7. Performance Monitoring

In [ ]:
import time

def benchmark_generation(prompt: str, max_tokens: int = 100):
    """Benchmark generation speed."""
    if not HARMONY_AVAILABLE:
        print("Harmony library required")
        return
    
    encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
    messages = [
        Message.from_role_and_content(Role.SYSTEM, SystemContent.new()),
        Message.from_role_and_content(Role.USER, prompt),
    ]
    conversation = Conversation.from_messages(messages)
    tokens = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
    
    start_time = time.time()
    
    with torch.no_grad():
        parser = StreamableParser(encoding, role=Role.ASSISTANT)
        input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
        
        tokens_generated = 0
        for _ in range(max_tokens):
            logits = model(input_ids)
            next_logits = logits[0, -1, :]
            next_token = torch.argmax(next_logits).item()
            
            parser.process(next_token)
            
            eos_tokens = {200002, 199999, 200012, 200007}
            if next_token in eos_tokens:
                break
            
            tokens_generated += 1
            input_ids = torch.cat([input_ids, torch.tensor([[next_token]], device=device)], dim=1)
    
    end_time = time.time()
    elapsed = end_time - start_time
    tokens_per_sec = tokens_generated / elapsed
    
    print(f"\nBenchmark Results:")
    print(f"  Tokens generated: {tokens_generated}")
    print(f"  Time elapsed: {elapsed:.2f}s")
    print(f"  Speed: {tokens_per_sec:.2f} tokens/sec")
    print(f"  Device: {device}")
    
    if torch.cuda.is_available():
        print(f"  GPU memory used: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

# Run benchmark
benchmark_generation(
    prompt="Explain how transformers work in machine learning.",
    max_tokens=100
)

## 8. Cleanup

In [ ]:
# Clear GPU memory
if torch.cuda.is_available():
    del model
    torch.cuda.empty_cache()
    print("✓ GPU memory cleared")
else:
    print("No GPU to clear")

## Notes

### Performance Tips:
- **GPU**: ~20-50 tokens/sec on A100
- **CPU**: ~2-5 tokens/sec (very slow)
- Use lower `temperature` (0.0-0.3) for deterministic/factual outputs
- Use higher `temperature` (0.7-1.0) for creative outputs

### Troubleshooting:
- **Out of memory**: Reduce batch size or use CPU
- **Slow inference**: Expected on CPU, use GPU if possible
- **Incoherent output**: Make sure Harmony library is installed

### Special Tokens:
- `<|start|>` (200006) - Message start
- `<|end|>` (200007) - Message end / EOS
- `<|message|>` (200008) - Content delimiter
- `<|return|>` (200002) - Alternative EOS
- `<|endoftext|>` (199999) - Text completion EOS